In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR SEQUENCE CLASSIFICATION (distilbert-base-uncased-finetuned-sst-2-english) ──
# Task         : Sentiment classification — Positive / Negative
# Architecture : DistilBERT — distilled (smaller, faster) version of BERT
#                6 transformer layers vs BERT's 12, ~40% smaller, ~60% faster
# Head         : Linear layer on top of [CLS] token → 2 output logits (NEGATIVE, POSITIVE)
# Fine-tuned on: SST-2 (Stanford Sentiment Treebank) — movie review sentences
# Output       : raw logits → softmax → probabilities → argmax → label
# ─────────────────────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# Same tokenizer as used during fine-tuning — vocab and special tokens must match
# AutoTokenizer inspects config.json in the model repo → picks the right tokenizer class
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

print("—" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  [CLS] id        : {tokenizer.cls_token_id}")
print(f"  [SEP] id        : {tokenizer.sep_token_id}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForSequenceClassification adds a classification head on top of the base model
# .from_pretrained() downloads:
#   config.json   → architecture details (layers, hidden size, num_labels, id2label)
#   pytorch_model.bin (or model.safetensors) → actual weights
# model.eval() → switches off dropout layers (training-only behaviour)
#   during training : dropout randomly zeros activations → regularisation
#   during inference: dropout must be OFF — you want deterministic, full output
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
model.eval()   # IMPORTANT — always call this before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num labels   : {model.config.num_labels}")
print(f"  id → label   : {model.config.id2label}")
#   id → label : {0: 'NEGATIVE', 1: 'POSITIVE'}


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# return_tensors="pt" → PyTorch tensors (not plain lists)
# produces: input_ids, attention_mask, (token_type_ids for BERT-based)
text = "This movie was absolutely fantastic!"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'fantastic', '!', '[SEP]']


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient computation for all ops inside the block
#   during training  : gradients are tracked so .backward() can update weights
#   during inference : you never call .backward(), so tracking wastes memory
#   no_grad reduces memory by ~50% and speeds up the forward pass
# model(**inputs) unpacks the dict → same as model(input_ids=..., attention_mask=...)
with torch.no_grad():
    outputs = model(**inputs)

# outputs is a SequenceClassifierOutput dataclass — a named container, not a plain tensor
# .logits → raw scores before any activation function, shape [batch_size, num_labels]
#   logit[0] = score for NEGATIVE
#   logit[1] = score for POSITIVE
#   higher score = model is more confident about that class
#   logits can be any real number (negative, zero, positive) — NOT probabilities yet
logits = outputs.logits

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type    : {type(outputs).__name__}")
print(f"  Logits shape   : {logits.shape}")       # torch.Size([1, 2])
print(f"  Raw logits     : {logits}")
#   tensor([[-3.4620,  3.6873]])  →  POSITIVE score >> NEGATIVE score


# ── STEP 5 : Post-process outputs ────────────────────────────────────────────
# softmax converts raw logits → probabilities that sum to 1.0
#   softmax(x_i) = exp(x_i) / sum(exp(x_j) for all j)
#   preserves relative order — highest logit → highest probability
#   dim=-1 → apply across last dimension (the num_labels axis)
probabilities = F.softmax(logits, dim=-1)

# argmax → index of the highest value → the predicted class id
predicted_class_id = logits.argmax(dim=-1).item()   # .item() → plain Python int

# model.config.id2label maps integer id → human-readable label string
predicted_label = model.config.id2label[predicted_class_id]

print("\nSTEP 5 — Post-processing")
print(f"  Probabilities  : {probabilities}")
#   tensor([[0.0006, 0.9994]])  →  0.06% NEGATIVE, 99.94% POSITIVE
print(f"  Predicted id   : {predicted_class_id}")   # 1
print(f"  Predicted label: {predicted_label}")       # POSITIVE

for class_id, label in model.config.id2label.items():
    print(f"    {label:10s} : {probabilities[0][class_id].item():.4f}")
#   NEGATIVE   : 0.0006
#   POSITIVE   : 0.9994


# ── AUTOMODEL VARIANTS — TASK REFERENCE ──────────────────────────────────────
# Class                                Head added on top of base model
# ─────────────────────────────────────────────────────────────────────────────
# AutoModel                            None — raw hidden states only
# AutoModelForSequenceClassification   Linear([CLS] → num_labels)
# AutoModelForTokenClassification      Linear(each token → num_labels) — NER, POS
# AutoModelForQuestionAnswering        Two linears — start logits + end logits per token
# AutoModelForMaskedLM                 Linear(each token → vocab_size) — fill [MASK]
# AutoModelForCausalLM                 Linear(each token → vocab_size) — next token
# AutoModelForSeq2SeqLM                Encoder + Decoder heads — translation, summarise
# AutoModelForMultipleChoice           Linear([CLS] per choice → 1 score) — MCQ
# ─────────────────────────────────────────────────────────────────────────────
# Rule : pick the class that matches your task
#        the head is the only part that changes — base transformer stays the same


# ── OUTPUT DATACLASS FIELDS — WHAT EACH VARIANT RETURNS ──────────────────────
# SequenceClassifierOutput   → .logits                    shape [B, num_labels]
# TokenClassifierOutput      → .logits                    shape [B, seq_len, num_labels]
# QuestionAnsweringOutput    → .start_logits, .end_logits shape [B, seq_len] each
# MaskedLMOutput             → .logits                    shape [B, seq_len, vocab_size]
# CausalLMOutput             → .logits                    shape [B, seq_len, vocab_size]
# BaseModelOutput            → .last_hidden_state         shape [B, seq_len, hidden_size]
# All variants optionally return:
#   .hidden_states  → tuple of tensors, one per layer (needs output_hidden_states=True)
#   .attentions     → tuple of attention weight matrices  (needs output_attentions=True)


# ── DRY RUN : text = "This movie was absolutely fantastic!" ──────────────────

# Stage 1 — Lowercase
# "This movie was absolutely fantastic!"
#  → "this movie was absolutely fantastic!"

# Stage 2 — tokenizer.tokenize() : WordPiece split, no specials, no IDs
# word          pieces              reason
# this       →  this               whole word in vocab
# movie      →  movie              whole word in vocab
# was        →  was                whole word in vocab
# absolutely →  absolutely         whole word in vocab
# fantastic  →  fantastic          whole word in vocab
# !          →  !                  punctuation split
#
# tokens = ['this', 'movie', 'was', 'absolutely', 'fantastic', '!']

# Stage 3 — tokenizer() : add specials + IDs
# pos  token        id
#  0   [CLS]       101
#  1   this        2023
#  2   movie       3185
#  3   was         2001
#  4   absolutely  7078
#  5   fantastic   10392
#  6   !            999
#  7   [SEP]        102
#
# input_ids      = tensor([[101, 2023, 3185, 2001, 7078, 10392, 999, 102]])
# attention_mask = tensor([[  1,    1,    1,    1,    1,     1,   1,   1]])

# Stage 4 — Forward pass (simplified)
# [CLS] token walks through 6 transformer layers
# each layer : self-attention → add & norm → feed-forward → add & norm
# after 6 layers: [CLS] hidden state shape = [1, 768]  (hidden_size = 768)
# classification head : Linear(768 → 2) applied to [CLS] hidden state
# raw logits = [[-3.4620, 3.6873]]
#               NEGATIVE  POSITIVE   → POSITIVE wins by large margin

# Stage 5 — Post-processing
# softmax([-3.4620, 3.6873])
#   exp(-3.4620) = 0.0314
#   exp( 3.6873) = 39.93
#   sum          = 39.96
#   probabilities = [0.0314/39.96,  39.93/39.96]
#                 = [0.0006,        0.9994]
# argmax         = 1   → id2label[1] = "POSITIVE"
# final answer   : POSITIVE  (99.94% confidence)

————————————————————————————————————————————————————————————
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 30,522
  [CLS] id        : 101
  [SEP] id        : 102


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class  : DistilBertForSequenceClassification
  Num labels   : 2
  id → label   : {0: 'NEGATIVE', 1: 'POSITIVE'}

STEP 3 — Input tokenized
  Raw text       : 'This movie was absolutely fantastic!'
  input_ids      : tensor([[  101,  2023,  3185,  2001,  7078, 10392,   999,   102]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'fantastic', '!', '[SEP]']

STEP 4 — Forward pass complete
  Output type    : SequenceClassifierOutput
  Logits shape   : torch.Size([1, 2])
  Raw logits     : tensor([[-4.3171,  4.6654]])

STEP 5 — Post-processing
  Probabilities  : tensor([[1.2557e-04, 9.9987e-01]])
  Predicted id   : 1
  Predicted label: POSITIVE
    NEGATIVE   : 0.0001
    POSITIVE   : 0.9999


In [4]:
from transformers import AutoTokenizer, AutoModel
import torch

# ── AUTOMODEL — RAW EMBEDDINGS (bert-base-uncased) ───────────────────────────
# Task         : No task — returns raw contextual representations only
# Architecture : BERT-base — 12 transformer layers, 12 attention heads
# Hidden size  : 768 — every token gets a 768-dimensional vector
# No head      : AutoModel has NO linear layer on top
#                → you get the full transformer body output, nothing more
#                → compare: AutoModelForSequenceClassification adds Linear(768→2)
# Use when     : you want embeddings for similarity, clustering, fine-tuning,
#                retrieval, or any custom downstream task
# Outputs      : .last_hidden_state → contextual vector per token
#                .pooler_output     → [CLS] vector through a Linear + Tanh
#                .hidden_states     → every layer's output (if requested)
#                .attentions        → every layer's attention weights (if requested)
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# attn_implementation="eager" — use classic attention instead of torch sdpa
# sdpa (scaled dot-product attention) — faster, fused kernel, but cannot return
#   attention weight matrices because the fused op discards them mid-computation
# eager — computes attention step by step, keeps all intermediate tensors
# ONLY needed when output_attentions=True — for all other steps sdpa is fine
# two separate model objects so normal inference stays fast (sdpa)
# and attention inspection uses eager only where required
model      = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

model_eager = AutoModel.from_pretrained(
    "bert-base-uncased",
    attn_implementation="eager"   # ← required for output_attentions=True
)
model_eager.eval()

print("\nSTEP 2 — Models loaded")
print(f"  model class       : {type(model).__name__}")
print(f"  attn (normal)     : sdpa  — fast, no attention weights")
print(f"  attn (eager)      : eager — slower, returns attention weights")
print(f"  Num layers        : {model.config.num_hidden_layers}")
print(f"  Hidden size       : {model.config.hidden_size}")
print(f"  Num heads         : {model.config.num_attention_heads}")
# UNEXPECTED keys warning — safe to ignore
#   bert-base-uncased checkpoint contains MLM + NSP head weights
#   (cls.predictions.*, cls.seq_relationship.*)
#   AutoModel loads only the transformer body — no heads
#   those weights have nowhere to go → flagged as UNEXPECTED
#   only a problem if you expected the full MLM model (use AutoModelForMaskedLM then)


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
text   = "Hello world"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'hello', 'world', '[SEP]']


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() — no gradient tracking, saves memory, speeds up inference
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type : {type(outputs).__name__}")


# ── STEP 5A : last_hidden_state ───────────────────────────────────────────────
# Shape — [batch_size, seq_len, hidden_size]
# One 768-dim vector per token, fully context-aware
# Every vector "knows" about all other tokens via self-attention
# [CLS] at index 0, [SEP] at index -1
last_hidden = outputs.last_hidden_state

print("\nSTEP 5A — last_hidden_state")
print(f"  Shape              : {last_hidden.shape}")
#   torch.Size([1, 4, 768])   — batch=1, tokens=4 ([CLS] hello world [SEP])
print(f"  [CLS] vector[:5]   : {last_hidden[0, 0, :5]}")
print(f"  'hello' vector[:5] : {last_hidden[0, 1, :5]}")
print(f"  'world' vector[:5] : {last_hidden[0, 2, :5]}")


# ── STEP 5B : pooler_output ───────────────────────────────────────────────────
# Takes the [CLS] hidden state (index 0) → passes through Linear(768→768) + Tanh
# Designed during BERT pre-training for next-sentence prediction
# One vector per sentence — shape [batch_size, hidden_size]
# NOTE — for sentence similarity tasks, mean pooling (below) usually works better
pooler = outputs.pooler_output

print("\nSTEP 5B — pooler_output")
print(f"  Shape          : {pooler.shape}")
#   torch.Size([1, 768])
print(f"  Vector[:5]     : {pooler[0, :5]}")


# ── STEP 5C : mean pooling (sentence embedding) ───────────────────────────────
# Why not just use pooler_output?
#   pooler_output uses only [CLS] — loses information from all other tokens
#   mean pooling averages ALL real token vectors → richer sentence representation
# Why use attention_mask?
#   padding tokens (mask=0) must be excluded from the average
#   without mask, padding zeros would drag all vectors toward zero
# Steps:
#   1. unsqueeze(-1)   — expand mask from [B, seq] to [B, seq, 1]
#   2. .expand(...)    — broadcast to [B, seq, 768] to match token_embeddings
#   3. multiply        — zero out padding positions in the embedding matrix
#   4. sum(dim=1)      — sum across token axis → [B, 768]
#   5. clamp + divide  — divide by real token count, clamp avoids division by zero
attention_mask   = inputs["attention_mask"]
token_embeddings = outputs.last_hidden_state
mask_expanded    = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
sentence_embedding = (
    torch.sum(token_embeddings * mask_expanded, dim=1)
    / torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
)

print("\nSTEP 5C — Mean pooling (sentence embedding)")
print(f"  attention_mask shape   : {attention_mask.shape}")
#   torch.Size([1, 4])
print(f"  mask_expanded shape    : {mask_expanded.shape}")
#   torch.Size([1, 4, 768])
print(f"  sentence_embedding shape : {sentence_embedding.shape}")
#   torch.Size([1, 768])
print(f"  Vector[:5]             : {sentence_embedding[0, :5]}")


# ── STEP 5D : all hidden states (every layer) ─────────────────────────────────
# output_hidden_states=True — returns output of every transformer layer
# .hidden_states — tuple of (num_layers + 1) tensors
#   index 0 — embedding layer output (token + position + segment embeddings summed)
#   index 1 — layer 1 output
#   ...
#   index 12 — layer 12 output (same as last_hidden_state)
# Each tensor shape — [batch_size, seq_len, hidden_size]
# Use case — layer-wise analysis, probing tasks, weighted-layer embeddings
with torch.no_grad():
    outputs_hidden = model(**inputs, output_hidden_states=True)

all_hidden = outputs_hidden.hidden_states

print("\nSTEP 5D — All hidden states")
print(f"  Num tensors in tuple : {len(all_hidden)}")
#   13  — embedding layer + 12 transformer layers
print(f"  Each tensor shape    : {all_hidden[0].shape}")
#   torch.Size([1, 4, 768])
for i, h in enumerate(all_hidden):
    label = "embedding" if i == 0 else f"layer {i:>2}"
    print(f"    {label} — {h.shape}")


# ── STEP 5E : attention weights (every layer, every head) ─────────────────────
# use model_eager here — sdpa model will return empty tuple ()
with torch.no_grad():
    outputs_attn = model_eager(**inputs, output_attentions=True)

all_attentions = outputs_attn.attentions

print("\nSTEP 5E — All attention weights")
print(f"  Num tensors in tuple : {len(all_attentions)}")
#   12 — one per transformer layer
print(f"  Each tensor shape    : {all_attentions[0].shape}")
#   torch.Size([1, 12, 4, 4]) — batch=1, heads=12, seq=4, seq=4
print(f"  Layer 1 head 0 attention matrix:")
print(f"  {all_attentions[0][0, 0]}")
#   each row sums to ~1.0 — softmax applied across key axis
for i, attn in enumerate(all_attentions):
    print(f"    layer {i+1:>2} — {attn.shape}")


# ── OUTPUT FIELDS — QUICK REFERENCE ──────────────────────────────────────────
# outputs.last_hidden_state   shape [B, seq_len, 768]  — always returned
# outputs.pooler_output       shape [B, 768]           — always returned
# outputs.hidden_states       tuple of 13 × [B, seq_len, 768]  — opt-in
# outputs.attentions          tuple of 12 × [B, heads, seq, seq] — opt-in
# ─────────────────────────────────────────────────────────────────────────────
# Choosing your embedding:
#   single token task  →  last_hidden_state[:, token_index, :]
#   sentence (fast)    →  pooler_output
#   sentence (better)  →  mean pooling over last_hidden_state
#   layer analysis     →  hidden_states[layer_index]
#   interpretability   →  attentions[layer_index]


# ── DRY RUN : text = "Hello world" ───────────────────────────────────────────

# Stage 1 — Tokenize
# "Hello world"
#  → lowercase → "hello world"
#  → WordPiece → ['hello', 'world']  (both in vocab, no subword splitting)
#  → add specials → ['[CLS]', 'hello', 'world', '[SEP]']
#  → IDs         → [101, 7592, 2088, 102]
#
# input_ids      = tensor([[101, 7592, 2088, 102]])
# attention_mask = tensor([[  1,    1,    1,   1]])    — no padding, all real tokens

# Stage 2 — Forward pass (simplified, one layer shown)
# Each of the 12 layers runs:
#   self-attention  — each token looks at all others, weighted by relevance
#   add & norm      — residual connection + LayerNorm for stability
#   feed-forward    — two linear layers with GELU activation
#   add & norm      — residual + LayerNorm again
# After layer 12:
#   last_hidden_state shape — [1, 4, 768]
#     index 0 → [CLS]   768-dim vector (context of full sentence)
#     index 1 → hello   768-dim vector (contextualised by "world")
#     index 2 → world   768-dim vector (contextualised by "hello")
#     index 3 → [SEP]   768-dim vector

# Stage 3 — pooler_output
# takes last_hidden_state[0, 0, :]   →   [CLS] vector, shape [768]
# → Linear(768 → 768) → Tanh
# → pooler_output shape [1, 768]

# Stage 4 — mean pooling
# mask_expanded zeros out nothing (no padding here — all mask values = 1)
# sum([CLS] + hello + world + [SEP]) / 4
# → sentence_embedding shape [1, 768]
# in practice [CLS] and [SEP] add noise — for cleaner embeddings strip them:
#   token_embeddings[:, 1:-1, :]  before mean pooling

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 30,522


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Models loaded
  model class       : BertModel
  attn (normal)     : sdpa  — fast, no attention weights
  attn (eager)      : eager — slower, returns attention weights
  Num layers        : 12
  Hidden size       : 768
  Num heads         : 12

STEP 3 — Input tokenized
  Raw text       : 'Hello world'
  input_ids      : tensor([[ 101, 7592, 2088,  102]])
  attention_mask : tensor([[1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'hello', 'world', '[SEP]']

STEP 4 — Forward pass complete
  Output type : BaseModelOutputWithPoolingAndCrossAttentions

STEP 5A — last_hidden_state
  Shape              : torch.Size([1, 4, 768])
  [CLS] vector[:5]   : tensor([-0.1689,  0.1361, -0.1394, -0.0544, -0.2953])
  'hello' vector[:5] : tensor([-0.3633,  0.1412,  0.8800, -1.0147, -0.1198])
  'world' vector[:5] : tensor([-0.6986, -0.6988,  0.0645, -0.4830,  0.1972])

STEP 5B — pooler_output
  Shape          : torch.Size([1, 768])
  Vector[:5]     : tensor([-0.9062, -0.3112, -0.6217,  0.7741,  0.2899]

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR SEQUENCE CLASSIFICATION (distilbert-base-uncased-finetuned-sst-2-english) ──
# Task         : Single-label text classification — one label per sentence
# Architecture : DistilBERT — 6 transformer layers, distilled from BERT's 12
#                ~40% fewer parameters, ~60% faster, ~97% of BERT's performance
# Head         : Linear(768 → num_labels) applied to [CLS] token only
#                [CLS] aggregates meaning of full sequence through self-attention
#                all other token outputs are discarded after the transformer body
# Fine-tuned on: SST-2 — Stanford Sentiment Treebank, movie review sentences
# num_labels   : 2 — NEGATIVE (0), POSITIVE (1)
# Output       : raw logits → softmax → probabilities → argmax → label string
# Use for      : sentiment analysis, topic classification, intent detection, spam
# ─────────────────────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# must match the tokenizer used during fine-tuning — vocab and special tokens must align
# AutoTokenizer reads config.json in the model repo → selects the correct class
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  [CLS] id        : {tokenizer.cls_token_id}")
print(f"  [SEP] id        : {tokenizer.sep_token_id}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForSequenceClassification — transformer body + classification head
# head — Linear(hidden_size → num_labels) on top of [CLS] hidden state
# model.eval() — disables dropout for deterministic inference output
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num labels   : {model.config.num_labels}")
print(f"  id2label     : {model.config.id2label}")
#   {0: 'NEGATIVE', 1: 'POSITIVE'}
print(f"  label2id     : {model.config.label2id}")
#   {'NEGATIVE': 0, 'POSITIVE': 1}


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# return_tensors="pt" — PyTorch tensors, required for model(**inputs)
# produces: input_ids, attention_mask
# DistilBERT does NOT use token_type_ids (no segment embeddings)
text   = "I love this product!"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'i', 'love', 'this', 'product', '!', '[SEP]']


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() — disables gradient computation
#   training   : gradients tracked → .backward() updates weights
#   inference  : gradients never needed → tracking wastes memory and compute
# model(**inputs) unpacks dict → model(input_ids=..., attention_mask=...)
# internally:
#   input_ids → embedding layer → 6 transformer layers → [CLS] hidden state
#   [CLS] hidden state → Linear(768 → 2) → raw logits
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type  : {type(outputs).__name__}")
#   SequenceClassifierOutput


# ── STEP 5 : Post-process — single-label classification ──────────────────────
# logits — raw scores, shape [batch_size, num_labels]
#   NOT probabilities — no activation applied yet
#   can be any real number (negative, zero, positive)
#   higher logit = model leans more toward that class
#   logit[0] = NEGATIVE score, logit[1] = POSITIVE score
logits = outputs.logits

print("\nSTEP 5 — Post-processing (single-label)")
print(f"  Logits shape   : {logits.shape}")
#   torch.Size([1, 2])
print(f"  Raw logits     : {logits}")
#   tensor([[-3.9, 4.1]])   — POSITIVE score dominates by large margin

# softmax — converts logits to probabilities that sum to 1.0
#   softmax(x_i) = exp(x_i) / Σ exp(x_j)
#   preserves relative order — highest logit → highest probability
#   use softmax for single-label (exactly one class is correct)
probs = F.softmax(logits, dim=-1)
print(f"  Probabilities  : {probs}")
#   tensor([[0.0002, 0.9998]])

# argmax — index of highest value → predicted class id
# .item() — converts 0-dim tensor to plain Python int
pred_id = logits.argmax(dim=-1).item()
label   = model.config.id2label[pred_id]
print(f"  Predicted id   : {pred_id}")
#   1
print(f"  Predicted label: {label}")
#   POSITIVE

print("\n  Per-class breakdown:")
for class_id, class_label in model.config.id2label.items():
    print(f"    {class_label:10s} — {probs[0][class_id].item():.4f}")
#   NEGATIVE   — 0.0002
#   POSITIVE   — 0.9998


# ── STEP 6 : Multi-label classification ──────────────────────────────────────
# single-label  — exactly one class is correct     → softmax (probs sum to 1.0)
# multi-label   — multiple classes can be correct  → sigmoid (each prob independent)
#
# sigmoid — squashes each logit independently to [0, 1]
#   sigmoid(x) = 1 / (1 + exp(-x))
#   each output is its own binary probability — unrelated to other outputs
#   no constraint that they sum to 1.0
#   threshold at 0.5 → above = class present, below = class absent
#
# example: topic tagging — a sentence can be [sports=0.9, news=0.8, politics=0.1]
# softmax would force them to compete — sigmoid lets each be independent
probs_multilabel = torch.sigmoid(logits)

print("\nSTEP 6 — Multi-label sigmoid (for reference)")
print(f"  sigmoid logits : {probs_multilabel}")
#   not meaningful here (SST-2 is single-label) — shown for format only
#   for real multi-label: classes above 0.5 threshold are predicted present
print(f"  Threshold 0.5  :")
for class_id, class_label in model.config.id2label.items():
    present = probs_multilabel[0][class_id].item() > 0.5
    print(f"    {class_label:10s} — {probs_multilabel[0][class_id].item():.4f}  →  {'present' if present else 'absent'}")


# ── SINGLE-LABEL vs MULTI-LABEL — DECISION GUIDE ─────────────────────────────
# single-label  : one correct answer per input   →  softmax + argmax
#   sentiment   : POSITIVE or NEGATIVE (not both)
#   intent      : book_flight or check_weather (not both)
#   topic       : sports or politics (mutually exclusive)
#
# multi-label   : multiple correct answers per input  →  sigmoid + threshold
#   tags        : a news article can be sports AND politics AND local
#   emotions    : a sentence can be sad AND angry simultaneously
#   genres      : a movie can be action AND comedy AND thriller
# ─────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : text = "I love this product!" ──────────────────────────────────

# Stage 1 — Tokenize
# "I love this product!"
#  → lowercase     → "i love this product!"
#  → WordPiece     → ['i', 'love', 'this', 'product', '!']  (all in vocab)
#  → add specials  → ['[CLS]', 'i', 'love', 'this', 'product', '!', '[SEP]']
#  → IDs           → [101, 1045, 2293, 2023, 4031, 999, 102]
#
# input_ids      = tensor([[101, 1045, 2293, 2023, 4031, 999, 102]])
# attention_mask = tensor([[  1,    1,    1,    1,    1,   1,   1]])   — no padding

# Stage 2 — Forward pass (simplified)
# token IDs → embedding layer → 6 DistilBERT transformer layers
# each layer: self-attention → add & norm → feed-forward → add & norm
# after layer 6: [CLS] hidden state shape [1, 768]
# classification head: Linear(768 → 2) applied to [CLS] only
# raw logits ≈ [[-3.9, 4.1]]
#               NEGATIVE  POSITIVE  → POSITIVE dominates

# Stage 3 — Post-processing
# softmax([-3.9, 4.1])
#   exp(-3.9) = 0.0202
#   exp( 4.1) = 60.34
#   sum        = 60.36
#   probs      = [0.0202/60.36, 60.34/60.36]
#              = [0.0002,       0.9998]
# argmax       = 1  →  id2label[1] = "POSITIVE"
# final answer : POSITIVE  (99.98% confidence)

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 30,522
  [CLS] id        : 101
  [SEP] id        : 102


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class  : DistilBertForSequenceClassification
  Num labels   : 2
  id2label     : {0: 'NEGATIVE', 1: 'POSITIVE'}
  label2id     : {'NEGATIVE': 0, 'POSITIVE': 1}

STEP 3 — Input tokenized
  Raw text       : 'I love this product!'
  input_ids      : tensor([[ 101, 1045, 2293, 2023, 4031,  999,  102]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'i', 'love', 'this', 'product', '!', '[SEP]']

STEP 4 — Forward pass complete
  Output type  : SequenceClassifierOutput

STEP 5 — Post-processing (single-label)
  Logits shape   : torch.Size([1, 2])
  Raw logits     : tensor([[-4.3599,  4.7156]])
  Probabilities  : tensor([[1.1443e-04, 9.9989e-01]])
  Predicted id   : 1
  Predicted label: POSITIVE

  Per-class breakdown:
    NEGATIVE   — 0.0001
    POSITIVE   — 0.9999

STEP 6 — Multi-label sigmoid (for reference)
  sigmoid logits : tensor([[0.0126, 0.9911]])
  Threshold 0.5  :
    NEGATIVE   — 0.0126  →  absent
    POSITIVE   — 0.99

In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR TOKEN CLASSIFICATION (dslim/bert-base-NER) ─────────────────
# Task         : Named Entity Recognition — label every token individually
# Architecture : BERT-base — 12 transformer layers, 12 attention heads, hidden 768
# Head         : Linear(768 → num_labels) applied to EVERY token (not just [CLS])
#                each token gets its own logit vector → its own label prediction
#                contrast with SequenceClassification — only [CLS] goes to head
# Fine-tuned on: CoNLL-2003 NER dataset — newswire text with 4 entity types
# num_labels   : 9 — O + B/I pairs for PER, ORG, LOC, MISC
# IOB2 scheme  : B- = beginning of entity span
#                I- = inside / continuation of entity span
#                O  = outside — not part of any entity
# Use for      : NER, POS tagging, chunking, any per-token labeling task
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# dslim/bert-base-NER was fine-tuned starting from bert-base-cased
# cased — capital letters preserved — critical for NER
# "Sarah" ≠ "sarah" — capitalisation is a strong signal for proper nouns
# using uncased here would hurt entity detection significantly
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  Cased           : {not tokenizer.do_lower_case}")
#   Cased : True — capital letters preserved, essential for NER


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForTokenClassification — transformer body + per-token linear head
# head — Linear(768 → num_labels) applied independently to every token position
# model.eval() — disables dropout for deterministic inference
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num labels   : {model.config.num_labels}")
print(f"  id2label     : {model.config.id2label}")
#   {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER',
#    5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
print(f"  label2id     : {model.config.label2id}")


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# cased model — preserves capitalisation exactly as written
# WordPiece may split rare words into subwords
#   e.g. "London" → ["London"]      (in vocab, no split)
#   e.g. "Schwarzenegger" → ["Sc", "##hwar", "##zen", "##eg", "##ger"]
# subword alignment problem — one word may produce multiple tokens
#   the model assigns a label to EACH subword token
#   common fix: take label of first subword, ignore ##-prefixed continuations
text   = "My name is Sarah and I live in London."
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"  Tokens         : {tokens}")
print(f"  Num tokens     : {len(tokens)}")
#   ['[CLS]', 'My', 'name', 'is', 'Sarah', 'and', 'I', 'live', 'in', 'London', '.', '[SEP]']
#   12 tokens total — [CLS] + 10 word tokens + . + [SEP]


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# every token passes through all 12 BERT layers
# each layer: self-attention → add & norm → feed-forward → add & norm
# after layer 12: hidden state shape [batch, seq_len, 768]
# classification head: Linear(768 → 9) applied to EVERY position
# result: logits shape [batch, seq_len, num_labels]
#         one 9-dim logit vector per token — one label predicted per token
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type  : {type(outputs).__name__}")
#   TokenClassifierOutput


# ── STEP 5 : Post-process — per-token labels ──────────────────────────────────
# logits shape — [batch_size, seq_len, num_labels]
#   axis 0 — batch (1 sentence here)
#   axis 1 — token position (12 tokens here)
#   axis 2 — label scores (9 NER labels)
#   each token has its own independent 9-dim logit vector
logits = outputs.logits

print("\nSTEP 5 — Post-processing")
print(f"  Logits shape  : {logits.shape}")
#   torch.Size([1, 12, 9])  — batch=1, seq=12, labels=9

# argmax along label axis → predicted class id for each token position
# [0] — first (only) sentence in batch
pred_ids = logits.argmax(dim=-1)[0]
print(f"  Pred id shape : {pred_ids.shape}")
#   torch.Size([12])  — one predicted id per token

# softmax along label axis → confidence score for predicted label
probs = F.softmax(logits, dim=-1)[0]

print("\n  Token — label predictions:")
print(f"  {'Token':15} {'Label':10} {'Confidence':>10}")
print(f"  {'─'*15} {'─'*10} {'─'*10}")
for token, pred_id, prob_vec in zip(tokens, pred_ids, probs):
    label      = model.config.id2label[pred_id.item()]
    confidence = prob_vec[pred_id].item()
    print(f"  {token:15} {label:10} {confidence:>10.4f}")
# [CLS]           O          0.9998
# My              O          0.9991
# name            O          0.9995
# is              O          0.9994
# Sarah           B-PER      0.9986     ← beginning of person entity
# and             O          0.9997
# I               O          0.9961
# live            O          0.9996
# in              O          0.9992
# London          B-LOC      0.9979     ← beginning of location entity
# .               O          0.9998
# [SEP]           O          0.9997


# ── STEP 6 : Entity span reconstruction ──────────────────────────────────────
# raw output gives one label per token — need to group into full entity spans
# IOB2 rules:
#   B-XXX → start a new entity of type XXX
#   I-XXX → continue the current entity (must follow B-XXX or I-XXX of same type)
#   O     → not part of any entity, close current span if one is open
# subword tokens (##-prefixed) inherit their word's label — skip them here
entities   = []
current    = None   # holds {"text": ..., "label": ..., "tokens": [...]}

for token, pred_id in zip(tokens, pred_ids):
    label = model.config.id2label[pred_id.item()]

    # skip special tokens — [CLS] and [SEP] are never real entities
    if token in ("[CLS]", "[SEP]"):
        continue

    # skip subword continuations — label belongs to the root token
    if token.startswith("##"):
        if current:
            current["tokens"].append(token[2:])   # strip ## and append to current word
        continue

    if label.startswith("B-"):
        # close previous entity if one was open
        if current:
            entities.append(current)
        # open new entity span
        current = {"text": token, "label": label[2:], "tokens": [token]}

    elif label.startswith("I-") and current and label[2:] == current["label"]:
        # continue current entity span — same type required
        current["tokens"].append(token)
        current["text"] = " ".join(current["tokens"])

    else:
        # O label or mismatched I- → close current span
        if current:
            entities.append(current)
            current = None

# close any entity still open at end of sequence
if current:
    entities.append(current)

print("\nSTEP 6 — Reconstructed entity spans:")
print(f"  {'Entity text':20} {'Type':10}")
print(f"  {'─'*20} {'─'*10}")
for ent in entities:
    print(f"  {ent['text']:20} {ent['label']:10}")
#   Sarah                PER
#   London               LOC


# ── IOB2 LABEL SCHEME — REFERENCE ────────────────────────────────────────────
# O        — Outside — not part of any named entity
# B-PER    — Beginning of a Person entity
# I-PER    — Inside / continuation of a Person entity
# B-ORG    — Beginning of an Organisation entity
# I-ORG    — Inside an Organisation entity
# B-LOC    — Beginning of a Location entity
# I-LOC    — Inside a Location entity
# B-MISC   — Beginning of a Miscellaneous entity (nationality, event, product)
# I-MISC   — Inside a Miscellaneous entity
# ─────────────────────────────────────────────────────────────────────────────
# Multi-token entity example:
#   "New  York  City  is  great"
#    B-LOC I-LOC I-LOC O   O
#   → one entity: "New York City" of type LOC
#   B starts the span — I continues it — O closes it


# ── SEQUENCE vs TOKEN CLASSIFICATION — KEY DIFFERENCE ─────────────────────────
# AutoModelForSequenceClassification
#   head applied to [CLS] token only → one label per sentence
#   logits shape — [batch, num_labels]
#
# AutoModelForTokenClassification
#   head applied to EVERY token → one label per token
#   logits shape — [batch, seq_len, num_labels]
#   argmax over dim=-1 to get predicted id per position
# ─────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : text = "My name is Sarah and I live in London." ─────────────────

# Stage 1 — Tokenize (cased — no lowercasing)
# "My name is Sarah and I live in London."
# word       →  token(s)      reason
# My         →  My            in vocab, cased
# name       →  name          in vocab
# is         →  is            in vocab
# Sarah      →  Sarah         in vocab, proper noun — capitalisation preserved
# and        →  and           in vocab
# I          →  I             in vocab
# live       →  live          in vocab
# in         →  in            in vocab
# London     →  London        in vocab, proper noun
# .          →  .             punctuation split
#
# final tokens with specials:
# ['[CLS]', 'My', 'name', 'is', 'Sarah', 'and', 'I', 'live', 'in', 'London', '.', '[SEP]']
# input_ids shape      — [1, 12]
# attention_mask shape — [1, 12]   all 1s, no padding

# Stage 2 — Forward pass
# each token walks through 12 BERT layers (self-attention + FFN each layer)
# output: last_hidden_state shape [1, 12, 768] — one 768-dim vector per token
# classification head: Linear(768 → 9) applied at every position
# logits shape: [1, 12, 9] — one 9-dim score vector per token

# Stage 3 — argmax over label axis
# token       logits (simplified)               argmax → label
# [CLS]       [3.1, -1.2, -1.1, ...]            0 → O
# My          [2.8, -0.9, -0.8, ...]            0 → O
# name        [3.0, -1.0, -0.9, ...]            0 → O
# is          [2.9, -1.1, -1.0, ...]            0 → O
# Sarah       [-2.1, -1.3, -0.8, 4.2, ...]      3 → B-PER
# and         [3.1, -0.9, -0.8, ...]            0 → O
# I           [2.7, -0.7, -0.6, ...]            0 → O
# live        [3.2, -1.0, -0.9, ...]            0 → O
# in          [3.0, -1.1, -1.0, ...]            0 → O
# London      [-1.8, -0.9, -0.7, -1.1, ..4.5.] 7 → B-LOC
# .           [3.2, -1.0, -0.9, ...]            0 → O
# [SEP]       [3.0, -0.9, -0.8, ...]            0 → O

# Stage 4 — Entity reconstruction
# Sarah → B-PER → open span {type: PER}
# and   → O     → close span → entity: "Sarah" PER
# London → B-LOC → open span {type: LOC}
# .     → O     → close span → entity: "London" LOC
# final entities: [("Sarah", PER), ("London", LOC)]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 28,996
  Cased           : True


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Model loaded
  Model class  : BertForTokenClassification
  Num labels   : 9
  id2label     : {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
  label2id     : {'B-LOC': 7, 'B-MISC': 1, 'B-ORG': 5, 'B-PER': 3, 'I-LOC': 8, 'I-MISC': 2, 'I-ORG': 6, 'I-PER': 4, 'O': 0}

STEP 3 — Input tokenized
  Raw text       : 'My name is Sarah and I live in London.'
  input_ids      : tensor([[ 101, 1422, 1271, 1110, 3696, 1105,  146, 1686, 1107, 1498,  119,  102]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'My', 'name', 'is', 'Sarah', 'and', 'I', 'live', 'in', 'London', '.', '[SEP]']
  Num tokens     : 12

STEP 4 — Forward pass complete
  Output type  : TokenClassifierOutput

STEP 5 — Post-processing
  Logits shape  : torch.Size([1, 12, 9])
  Pred id shape : torch.Size([12])

  Token — label predictions:
  Token           Label      Confidence
  ─────────────── ────────── ──────────

In [7]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR QUESTION ANSWERING (deepset/roberta-base-squad2) ────────────
# Task         : Extractive QA — find the answer span inside a given context
#                does NOT generate answers — only points to where the answer is
# Architecture : RoBERTa-base — 12 transformer layers, 12 heads, hidden 768
#                RoBERTa = BERT retrained longer, larger batches, no NSP task
#                no token_type_ids — RoBERTa dropped segment embeddings
# Head         : two separate Linear(768 → 1) layers — one for start, one for end
#                each produces one scalar score per token position
#                argmax over seq_len axis → predicted start and end index
# Fine-tuned on: SQuAD 2.0 — includes unanswerable questions
#                model learns to output [CLS] as start when no answer exists
# Output       : start_logits [B, seq_len] + end_logits [B, seq_len]
# Use for      : reading comprehension, document QA, FAQ extraction
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# RoBERTa uses BPE (Byte-Pair Encoding) — different from BERT's WordPiece
# BPE — merges frequent byte pairs iteratively to build vocab
#   BERT WordPiece : "playing" → ["play", "##ing"]   (## marks continuation)
#   RoBERTa BPE    : "playing" → ["play", "ing"]     (no ## marker, uses Ġ for spaces)
# Vocab size 50,265 — larger than BERT's 30,522
# Cased by default — RoBERTa preserves capitalisation
tokenizer = AutoTokenizer.from_pretrained("deepset/roberta-base-squad2")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  [CLS] token     : {tokenizer.cls_token!r}  id — {tokenizer.cls_token_id}")
print(f"  [SEP] token     : {tokenizer.sep_token!r}  id — {tokenizer.sep_token_id}")
#   RoBERTa uses <s> and </s> instead of [CLS] and [SEP]


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForQuestionAnswering — transformer body + two linear heads
# start head — Linear(768 → 1) per token → start_logits [B, seq_len]
# end head   — Linear(768 → 1) per token → end_logits   [B, seq_len]
# both heads are independent — end position is NOT conditioned on start
# model.eval() — disables dropout for deterministic inference
model = AutoModelForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num layers   : {model.config.num_hidden_layers}")
print(f"  Hidden size  : {model.config.hidden_size}")
print(f"  Num heads    : {model.config.num_attention_heads}")


# ── STEP 3 : Tokenize question + context as a pair ────────────────────────────
# QA requires two inputs: question and context (passage to search for answer)
# tokenizer(question, context) — sentence-pair encoding
#   RoBERTa format: <s> question </s> </s> context </s>
#   BERT format   : [CLS] question [SEP] context [SEP]
# model learns to read the question first, then scan context for the answer span
# question is always the first argument — this is convention, must not be swapped
question = "Where does Sarah live?"
context  = "My name is Sarah and I live in London, England."
inputs   = tokenizer(question, context, return_tensors="pt")

print("\nSTEP 3 — Input tokenized (question + context pair)")
print(f"  Question       : {question!r}")
print(f"  Context        : {context!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"  Tokens         : {tokens}")
print(f"  Num tokens     : {len(tokens)}")
#   ['<s>', 'Where', 'Ġdoes', 'ĠSarah', 'Ġlive', '?', '</s>', '</s>',
#    'ĠMy', 'Ġname', 'Ġis', 'ĠSarah', 'Ġand', 'ĠI', 'Ġlive', 'Ġin',
#    'ĠLondon', ',', 'ĠEngland', '.', '</s>']
#   Ġ prefix — RoBERTa BPE marker meaning "preceded by a space"
#   double </s></s> — RoBERTa's sentence boundary marker (BERT uses single [SEP])


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# all tokens attend to all other tokens across question AND context
# this lets context tokens "see" what the question is asking
# after 12 layers: hidden state shape [1, seq_len, 768] — one vector per token
# start head: Linear(768 → 1) at every position → start_logits [1, seq_len]
# end head  : Linear(768 → 1) at every position → end_logits   [1, seq_len]
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
#   QuestionAnsweringModelOutput
print(f"  start_logits shape : {outputs.start_logits.shape}")
print(f"  end_logits shape   : {outputs.end_logits.shape}")
#   both — torch.Size([1, 21])  — one score per token position


# ── STEP 5 : Extract answer span ─────────────────────────────────────────────
# start_logits — score for each token being the answer start position
# end_logits   — score for each token being the answer end position
# argmax independently on each → most likely start and end index
# NOTE — no constraint that end >= start here; for robust decoding see Step 7
start_logits = outputs.start_logits
end_logits   = outputs.end_logits

start_pos = start_logits.argmax(dim=-1).item()
end_pos   = end_logits.argmax(dim=-1).item()

# slice the token IDs from start to end (inclusive) → decode to string
all_tokens = inputs["input_ids"][0]
answer_ids = all_tokens[start_pos : end_pos + 1]
answer     = tokenizer.decode(answer_ids, skip_special_tokens=True)

print("\nSTEP 5 — Answer span extracted")
print(f"  start_pos      : {start_pos}  → token {tokens[start_pos]!r}")
print(f"  end_pos        : {end_pos}    → token {tokens[end_pos]!r}")
print(f"  answer_ids     : {answer_ids.tolist()}")
print(f"  Answer         : {answer!r}")
#   Answer : 'London, England'


# ── STEP 6 : Confidence scores ────────────────────────────────────────────────
# softmax over seq_len axis → probability that each position is start/end
# NOT the same as class probabilities (no competing labels here)
# these probabilities say: "of all token positions, how likely is each one?"
start_probs = F.softmax(start_logits, dim=-1)[0]
end_probs   = F.softmax(end_logits,   dim=-1)[0]

# answer confidence — product of start and end probabilities at chosen positions
answer_score = (start_probs[start_pos] * end_probs[end_pos]).item()

print("\nSTEP 6 — Confidence scores")
print(f"  Start prob at pos {start_pos:>2}  : {start_probs[start_pos].item():.4f}")
print(f"  End prob at pos   {end_pos:>2}  : {end_probs[end_pos].item():.4f}")
print(f"  Answer score       : {answer_score:.4f}")
#   product of start × end prob — rough overall confidence

print("\n  Top 5 start positions:")
top_start = start_probs.topk(5)
for score, idx in zip(top_start.values, top_start.indices):
    print(f"    pos {idx.item():>2} — {tokens[idx.item()]:15} — {score.item():.4f}")

print("\n  Top 5 end positions:")
top_end = end_probs.topk(5)
for score, idx in zip(top_end.values, top_end.indices):
    print(f"    pos {idx.item():>2} — {tokens[idx.item()]:15} — {score.item():.4f}")


# ── STEP 7 : Unanswerable questions (SQuAD 2.0) ──────────────────────────────
# SQuAD 2.0 — half of questions have no answer in the context
# model trained to output <s> / [CLS] (position 0) when no answer exists
# position 0 = <s> token — never a valid answer token
# so: if argmax lands on position 0 → model says "no answer found"
#
# null score — logit score when model predicts no answer:
#   null_score = start_logits[0][0] + end_logits[0][0]   (both point to [CLS])
# answer score — logit score for the best valid span:
#   answer_score = start_logits[0][start_pos] + end_logits[0][end_pos]
# decision: answer_score > null_score → answer exists, else no answer
null_score   = (start_logits[0][0] + end_logits[0][0]).item()
span_score   = (start_logits[0][start_pos] + end_logits[0][end_pos]).item()

print("\nSTEP 7 — Unanswerable check (SQuAD 2.0 style)")
print(f"  Null score (pos 0 + pos 0) : {null_score:.4f}")
print(f"  Span score (start + end)   : {span_score:.4f}")
if span_score > null_score:
    print(f"  Decision — answer found    : {answer!r}")
else:
    print(f"  Decision — no answer found in context")
#   span_score > null_score → answer exists → "London, England"


# ── SEQUENCE vs TOKEN vs QA — OUTPUT SHAPE COMPARISON ────────────────────────
# AutoModelForSequenceClassification
#   logits — [B, num_labels]            one vector per sentence
#   argmax over dim=-1 → predicted class
#
# AutoModelForTokenClassification
#   logits — [B, seq_len, num_labels]   one vector per token
#   argmax over dim=-1 → predicted class per token
#
# AutoModelForQuestionAnswering
#   start_logits — [B, seq_len]         one scalar per token
#   end_logits   — [B, seq_len]         one scalar per token
#   argmax over dim=-1 → start index, end index → slice → decode
# ─────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : question="Where does Sarah live?" context="My name is Sarah..." ─

# Stage 1 — Tokenize as sentence pair
# RoBERTa format: <s> question </s></s> context </s>
# token       pos   note
# <s>          0    sentence start (= [CLS] in BERT)
# Where        1    question token
# Ġdoes        2    Ġ = space before "does" in BPE
# ĠSarah       3    question token
# Ġlive        4    question token
# ?            5    punctuation
# </s>         6    sentence separator
# </s>         7    RoBERTa double separator between segments
# ĠMy          8    context starts here
# Ġname        9
# Ġis         10
# ĠSarah      11
# Ġand        12
# ĠI          13
# Ġlive       14
# Ġin         15
# ĠLondon     16   ← answer start
# ,           17
# ĠEngland    18   ← answer end
# .           19
# </s>        20   context end

# Stage 2 — Forward pass
# cross-attention between question and context:
#   "live" in question attends strongly to "live" in context (pos 14)
#   "Sarah" in question attends to "Sarah" in context (pos 11)
# start head scores — peak at pos 16 (ĠLondon)
# end head scores   — peak at pos 18 (ĠEngland)

# Stage 3 — Span extraction
# start_pos = 16 → ĠLondon
# end_pos   = 18 → ĠEngland
# answer_ids = input_ids[16:19]
# tokenizer.decode(answer_ids) → "London, England"

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : RobertaTokenizer
  Vocab size      : 50,265
  [CLS] token     : '<s>'  id — 0
  [SEP] token     : '</s>'  id — 2


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Model loaded
  Model class  : RobertaForQuestionAnswering
  Num layers   : 12
  Hidden size  : 768
  Num heads    : 12

STEP 3 — Input tokenized (question + context pair)
  Question       : 'Where does Sarah live?'
  Context        : 'My name is Sarah and I live in London, England.'
  input_ids      : tensor([[    0, 13841,   473,  4143,   697,   116,     2,     2,  2387,   766,
            16,  4143,     8,    38,   697,    11,   928,     6,  1156,     4,
             2]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['<s>', 'Where', 'Ġdoes', 'ĠSarah', 'Ġlive', '?', '</s>', '</s>', 'My', 'Ġname', 'Ġis', 'ĠSarah', 'Ġand', 'ĠI', 'Ġlive', 'Ġin', 'ĠLondon', ',', 'ĠEngland', '.', '</s>']
  Num tokens     : 21

STEP 4 — Forward pass complete
  Output type      : QuestionAnsweringModelOutput
  start_logits shape : torch.Size([1, 21])
  end_logits shape   : torch.Size([1, 21])

STEP 5 — Answer span extracted
  start_